In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
from textblob import TextBlob
from datetime import datetime, time
import pytz
import webbrowser
import os


In [2]:
# CELL 2 - Load the datasets
apps_path = r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\apps.csv"
reviews_path = r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\user_reviews.csv"

apps_df = pd.read_csv(apps_path)
reviews_df = pd.read_csv(reviews_path)

print("Apps shape:", apps_df.shape)
print("Reviews shape:", reviews_df.shape)
display(apps_df.head())
display(reviews_df.head())


Apps shape: (9659, 14)
Reviews shape: (64295, 5)


,Unnamed: 0,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [3]:
# CELL 3 - Clean and convert required columns
apps_df = apps_df.copy()
reviews_df = reviews_df.copy()

apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')

apps_df['Installs'] = pd.to_numeric(
    apps_df['Installs'].astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False),
    errors='coerce'
)

def convert_size(size):
    if pd.isna(size):
        return np.nan
    size = str(size).strip()
    if size.lower() == 'varies with device':
        return np.nan
    if size.endswith('M'):
        return pd.to_numeric(size[:-1], errors='coerce')
    if size.endswith(('k', 'K')):
        value = pd.to_numeric(size[:-1], errors='coerce')
        return value / 1024 if pd.notna(value) else np.nan
    return pd.to_numeric(size, errors='coerce')

apps_df['Size_MB'] = apps_df['Size'].apply(convert_size)

apps_df = apps_df.drop_duplicates(subset=['App']).copy()
apps_df = apps_df.dropna(
    subset=['App', 'Rating', 'Reviews', 'Installs', 'Size_MB', 'Category']
)
apps_df = apps_df[(apps_df['Rating'] >= 0) & (apps_df['Rating'] <= 5)].copy()

print("Cleaned apps:", apps_df.shape)
display(apps_df[['App','Category','Rating','Reviews','Installs','Size_MB']].head())


Cleaned apps: (7027, 15)


,App,Category,Rating,Reviews,Installs,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,10000,19.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967,500000,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,5000000,8.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,50000000,25.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,100000,2.8


In [4]:
# CELL 4 - Calculate review sentiment subjectivity
# TextBlob subjectivity is between 0 and 1.

reviews_df = reviews_df.dropna(
    subset=['App', 'Translated_Review']
).copy()

def get_subjectivity(text):
    return TextBlob(str(text)).sentiment.subjectivity

reviews_df['Sentiment_Subjectivity'] = (
    reviews_df['Translated_Review'].apply(get_subjectivity)
)

subjectivity_by_app = (
    reviews_df.groupby('App', as_index=False)['Sentiment_Subjectivity']
    .mean()
)

display(subjectivity_by_app.head())


,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.545516
2,11st,0.455340
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


In [5]:
# CELL 5 - Merge app data with review subjectivity
bubble_df = pd.merge(
    apps_df,
    subjectivity_by_app,
    on='App',
    how='inner'
)

print("Merged shape:", bubble_df.shape)


Merged shape: (568, 16)


In [6]:
# CELL 6 - Apply ALL requested filters

allowed_categories = [
    'GAME',
    'BEAUTY',
    'BUSINESS',
    'COMICS',
    'COMMUNICATION',
    'DATING',
    'ENTERTAINMENT',
    'SOCIAL',
    'EVENTS'
]

filtered_df = bubble_df[
    (bubble_df['Rating'] > 3.5) &
    (bubble_df['Category'].str.upper().isin(allowed_categories)) &
    (bubble_df['Reviews'] > 500) &
    (~bubble_df['App'].str.contains('S', case=False, na=False)) &
    (bubble_df['Sentiment_Subjectivity'] > 0.5) &
    (bubble_df['Installs'] > 50000)
].copy()

print("Apps after all filters:", len(filtered_df))
display(
    filtered_df[
        ['App','Category','Rating','Reviews','Installs',
         'Size_MB','Sentiment_Subjectivity']
    ].head(20)
)


Apps after all filters: 23


,App,Category,Rating,Reviews,Installs,Size_MB,Sentiment_Subjectivity
30,Google Primer,BUSINESS,4.4,62272,10000000,18.0,0.675000
31,Call Blocker,BUSINESS,4.6,188841,5000000,3.2,0.655431
54,"CallApp: Caller ID, Blocker & Phone Call Recorder",COMMUNICATION,4.4,483565,10000000,20.0,0.506481
56,Caller ID +,COMMUNICATION,4.0,9498,1000000,0.1,0.600000
59,"Hily: Dating, Chat, Match, Meet & Hook up",DATING,4.1,2556,100000,56.0,0.555331
62,CMB Free Dating App,DATING,4.0,48845,1000000,40.0,0.542165
78,FlirtChat - ♥Free Dating/Flirting App♥,DATING,4.3,2430,500000,13.0,0.510696
85,Hitwe - meet people and chat,DATING,4.2,243950,10000000,21.0,0.694105
86,"2Date Dating App, Love and matching",DATING,4.4,41605,500000,8.1,0.558391
105,Amazon Prime Video,ENTERTAINMENT,4.2,411683,50000000,24.0,0.555556


In [7]:
# CELL 7 - Translate category labels for the graph
# Beauty -> Hindi
# Business -> Tamil
# Dating -> German

category_translation = {
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Partnersuche'
}

filtered_df['Graph_Category'] = (
    filtered_df['Category']
    .str.upper()
    .map(lambda x: category_translation.get(x, x.title()))
)

display(
    filtered_df[['Category','Graph_Category']]
    .drop_duplicates()
    .sort_values('Category')
)


,Category,Graph_Category
30,BUSINESS,வணிகம்
54,COMMUNICATION,Communication
59,DATING,Partnersuche
105,ENTERTAINMENT,Entertainment
237,GAME,Game
363,SOCIAL,Social


In [8]:
# CELL 8 - Allow the graph only from 5 PM to 7 PM IST

IST = pytz.timezone('Asia/Kolkata')
current_ist = datetime.now(IST)
current_time = current_ist.time()

start_time = time(17, 0)  # 5:00 PM
end_time = time(19, 0)    # 7:00 PM

graph_allowed = start_time <= current_time < end_time

print(
    "Current IST:",
    current_ist.strftime('%Y-%m-%d %I:%M:%S %p')
)
print("Graph allowed:", graph_allowed)


Current IST: 2026-09-07 07:57:20 PM
Graph allowed: False


In [10]:
# CELL 9 - Create the bubble chart
# Game category is highlighted in PINK.

if not graph_allowed:
    print(
        "Bubble chart is hidden. It is available only "
        "between 5:00 PM and 7:00 PM IST."
    )

elif filtered_df.empty:
    print("No apps satisfy all the requested filters.")

else:
    unique_categories = filtered_df['Graph_Category'].unique().tolist()

    # Default colour for other categories.
    color_map = {
        category: '#9C27B0'
        for category in unique_categories
    }

    # Game must be pink.
    color_map['Game'] = '#FF1493'

    fig = px.scatter(
        filtered_df,
        x='Size_MB',
        y='Rating',
        size='Installs',
        color='Graph_Category',
        hover_name='App',
        hover_data={
            'Size_MB': ':.2f',
            'Rating': ':.2f',
            'Installs': ':,',
            'Reviews': ':,',
            'Sentiment_Subjectivity': ':.2f',
            'Graph_Category': True
        },
        labels={
            'Size_MB': 'App Size (MB)',
            'Rating': 'Average Rating',
            'Installs': 'Number of Installs',
            'Graph_Category': 'Category'
        },
        title='App Size vs Average Rating - Bubble Size = Installs',
        color_discrete_map=color_map,
        size_max=55,
        opacity=0.75
    )

    fig.update_traces(
        marker=dict(line=dict(width=1, color='white'))
    )

    fig.update_layout(
        width=1000,
        height=650,
        template='plotly_dark',
        title_font=dict(size=20),
        xaxis=dict(title='App Size (MB)'),
        yaxis=dict(
            title='Average Rating',
            range=[3.5, 5.05]
        ),
        legend_title='Category',
        margin=dict(l=60, r=40, t=80, b=60)
    )

    fig.show()


Bubble chart is hidden. It is available only between 5:00 PM and 7:00 PM IST.


In [11]:
# CELL 10 - Interactive category filter/dropdown

if graph_allowed and not filtered_df.empty:

    fig_filter = px.scatter(
        filtered_df,
        x='Size_MB',
        y='Rating',
        size='Installs',
        color='Graph_Category',
        hover_name='App',
        hover_data=[
            'Reviews',
            'Installs',
            'Sentiment_Subjectivity'
        ],
        labels={
            'Size_MB': 'App Size (MB)',
            'Rating': 'Average Rating',
            'Installs': 'Number of Installs',
            'Graph_Category': 'Category'
        },
        title='Filtered Play Store Bubble Chart',
        color_discrete_map=color_map,
        size_max=55,
        opacity=0.75
    )

    category_list = [
        'All Categories'
    ] + sorted(filtered_df['Graph_Category'].unique().tolist())

    buttons = []

    for category in category_list:

        if category == 'All Categories':
            visibility = [True] * len(fig_filter.data)
        else:
            visibility = [
                trace.name == category
                for trace in fig_filter.data
            ]

        buttons.append({
            'label': category,
            'method': 'update',
            'args': [
                {'visible': visibility},
                {
                    'title':
                    f'Filtered Play Store Bubble Chart - {category}'
                }
            ]
        })

    fig_filter.update_layout(
        template='plotly_dark',
        width=1000,
        height=650,
        updatemenus=[{
            'buttons': buttons,
            'direction': 'down',
            'showactive': True,
            'x': 1.0,
            'xanchor': 'right',
            'y': 1.15,
            'yanchor': 'top'
        }]
    )

    fig_filter.show()

else:
    print(
        "Category filter is hidden because the graph is outside "
        "the allowed time window or no data passed the filters."
    )


Category filter is hidden because the graph is outside the allowed time window or no data passed the filters.


In [12]:
# CELL 11 - Save the dashboard graph as HTML
# It is created ONLY during the allowed 5 PM - 7 PM IST window.

dashboard_path = os.path.abspath(
    'play_store_bubble_chart.html'
)

if graph_allowed and not filtered_df.empty:
    fig_filter.write_html(
        dashboard_path,
        include_plotlyjs='inline'
    )

    print("Dashboard saved to:", dashboard_path)
    webbrowser.open(
        'file://' + dashboard_path
    )
else:
    print(
        "Dashboard graph was not created because the allowed "
        "time is 5:00 PM to 7:00 PM IST only."
    )


Dashboard graph was not created because the allowed time is 5:00 PM to 7:00 PM IST only.
